# 03 - Tuning

One loop per target family, alternating two settings that interact, until they
stop moving each other:

```
round 1   window over the full (L, alpha) grid, at default parameters
          hyperparameter descent from that window
round 2   window over a neighbourhood of the incumbent, at ROUND-1 parameters
          descent from that window, continuing where round 1 finished
  ...     until a round moves neither, or MAX_TUNING_ROUNDS is reached
```

The three fixed stages this replaces ran once each and ended in a dead end: the
stability re-check updated the window when the winner moved, and then froze
hyperparameters that had been tuned at the *old* window. It was detecting the
interaction and had no way to act on it.

**The window is searched on the selected features**, not on `CORE_FEATURES`.
Smoothing may or may not be independent of the feature set, but choosing it
against ten hardcoded columns meant choosing it for a model nobody fits.

**The most recent development season is withheld.** Tuning sees the folds before
it; the settled configuration is scored against it once, at the end. The gap is
the only read on whether extra freedom bought signal or overfitting that does not
cost the test season, which is looked at once ever.

**`n_estimators` is a ceiling, not a budget.** Early stopping picks the count for
each candidate, so a learning rate is judged at the number of trees it wants --
measured, 0.01 wants ~994 and 0.05 wants ~213, and the old fixed 400 granted the
second and denied the first. `max_bin` is a tuned axis for the same reason: it
used to arrive from the probe config and get frozen at 64 without anything
choosing it.

Test seasons are untouched throughout.

In [1]:
import fpp
import numpy as np
import pandas as pd

pd.set_option("display.max_columns", None)
print("fpp", fpp.__version__, "| targets:", fpp.TARGETS)

fpp 0.1.0 | targets: ('goals', 'shots', 'sot', 'corners')


## 1. Config

In [2]:
from fpp.config import (
    ALPHA_GRID, DESCENT_PASSES, L_GRID, MAX_TUNING_ROUNDS, SEARCH,
    holdout_season,
)

ROUNDS = MAX_TUNING_ROUNDS   # cap on the alternating loop; it stops early if it settles
PASSES = DESCENT_PASSES      # descent passes per round; a pass that moves nothing ends it

df = fpp.clean.load_clean_table()
bw = fpp.BufferWindows(df, L_max=max(L_GRID))

# The version being *built*, not the one currently frozen. Resolving this from the
# manifest's "current" would make Tuning read and overwrite whatever was last
# frozen, silently ignoring the features Features just selected.
VERSION = fpp.artifacts.working_version()
print("artifact version:", VERSION, "| frozen:", fpp.artifacts.read_manifest().get("current"))

missing = [t for t in fpp.TARGETS
           if not (fpp.artifacts.artifact_dir(t, VERSION) / "features.json").exists()]
assert not missing, (
    f"No features artifact at {VERSION} for {missing}. "
    "Run 02_Features first, or set FPP_VERSION to the version you mean to tune."
)

print(f"window grid : {len(L_GRID)} x {len(ALPHA_GRID)} = {len(L_GRID) * len(ALPHA_GRID)} points")
print(f"loop        : up to {ROUNDS} rounds x {PASSES} descent passes")
print(f"folds       : tuning up to the season before {holdout_season()}, "
      f"{holdout_season()} withheld")
print(f"fit         : {SEARCH}")

artifact version: v2026-08-25 | frozen: v2026-08-21
window grid : 29 x 9 = 261 points
loop        : up to 3 rounds x 2 descent passes
folds       : tuning up to the season before 2024/2025, 2024/2025 withheld
fit         : FitConfig(learning_rate=0.05, n_estimators=3000, early_stopping_rounds=50, max_bin=64, nthread=1)


## 2. Tune

Expect 2-4 hours for all four families. Each round prints what moved; a run that
never settles says so rather than quietly returning its last state.

In [3]:
RUN_TUNING = True

tuned = {}
for t in fpp.TARGETS:
    feats = fpp.artifacts.load_artifact(t, "features", VERSION)["selected"]
    print(f"\n{'=' * 70}\n{t}  ({len(feats)} features)\n{'=' * 70}")
    if RUN_TUNING:
        tuned[t] = fpp.search.run_tuning(df, bw, t, feats, rounds=ROUNDS, passes=PASSES)
    else:
        w = fpp.artifacts.load_artifact(t, "window", VERSION)
        p = fpp.artifacts.load_artifact(t, "params", VERSION)
        tuned[t] = {"window": w, "params": p["params"], "logloss": float("nan"),
                    "logloss_se": float("nan"), "best_iterations": [],
                    "dispersion": p.get("dispersion", {}), "rounds": [], "converged": None}


goals  (5 features)
  [goals] tuning on 4 folds (2020/2021 .. 2023/2024); 2024/2025 withheld

  [goals] --- round 1 ---
[goals/window] 261 points, 261 cached, 0 to run
  [goals] window -> L= 50 alpha=0.05  ll=1.458738  (effective window 50 matches)
  [goals]   outright best was L= 85 alpha=0.05 ll=1.457938 -- plateau entry costs +0.000800 (+0.23 SE) for 35 fewer matches of history
  [goals] p0 learning_rate              -> {'learning_rate': 0.05}  ll=1.458738  margin +0.000000 (+0.00 SE)  [<= tol 0.001, kept {'learning_rate': 0.05}]
  [goals] p0 max_depth+min_child_weight -> {'max_depth': 2, 'min_child_weight': 30}  ll=1.458425
  [goals] p0 subsample+colsample_bytree -> {'subsample': 0.5, 'colsample_bytree': 0.6}  ll=1.457520
  [goals] p0 reg_alpha+reg_lambda       -> {'reg_alpha': 0, 'reg_lambda': 1}  ll=1.457520
  [goals] p0 max_bin                    -> {'max_bin': 64}  ll=1.457520  margin +0.000000 (+0.00 SE)  [<= tol 0.001, kept {'max_bin': 64}]
  [goals] p1 learning_rate        

## 3. What each round moved

A target still moving at the last round has not settled, and its numbers are the
state the loop happened to stop in rather than a resting point.

In [4]:
rows = []
for t, r in tuned.items():
    for h in r["rounds"]:
        rows.append({"target": t, "round": h["round"], "L": h["window"]["L"],
                     "alpha": h["window"]["alpha"], "logloss": h["logloss"],
                     "window": "moved" if h["window_moved"] else "held",
                     "params": "moved" if h["params_moved"] else "held"})
if rows:
    display(pd.DataFrame(rows).set_index(["target", "round"]))
for t, r in tuned.items():
    if r["converged"] is False:
        print(f"  {t}: did NOT converge -- window and learner still moving each other")

L  alpha   logloss window params
target  round                                   
goals   1      50   0.05  1.457501  moved  moved
        2      41   0.02  1.458409  moved   held
        3      41   0.02  1.458409   held   held
shots   1      60   0.10  2.844932  moved  moved
        2      50   0.10  2.845520  moved   held
        3      50   0.10  2.845520   held   held
sot     1      60   0.05  2.160562  moved  moved
        2      60   0.05  2.160562   held   held
corners 1      35   0.05  2.327323  moved  moved
        2      35   0.05  2.327323   held   held

## 4. The withheld season

`gap` is the holdout loss minus the tuning-CV loss. Positive means the tuned
configuration does worse on a season it never saw than on the ones it searched --
which is what buying an improvement with dials rather than signal looks like.

Read it as a direction, not a verdict: one season is one number, and a small gap
either way is inside the noise of a single fold.

In [5]:
holdout = {}
for t in fpp.TARGETS:
    feats = fpp.artifacts.load_artifact(t, "features", VERSION)["selected"]
    h = fpp.search.holdout_score(df, bw, t, feats, tuned[t]["window"], tuned[t]["params"])
    h["tuning_cv"] = tuned[t]["logloss"]
    h["gap"] = h["logloss"] - tuned[t]["logloss"]
    holdout[t] = h

display(pd.DataFrame(holdout).T[["season", "tuning_cv", "logloss", "gap"]]
        .rename(columns={"logloss": "holdout"}))

,season,tuning_cv,holdout,gap
goals,2024/2025,1.458409,1.453658,-0.004751
shots,2024/2025,2.84552,2.853777,0.008257
sot,2024/2025,2.160562,2.140824,-0.019738
corners,2024/2025,2.327323,2.324402,-0.002921


## 5. Freeze

In [6]:
for t in fpp.TARGETS:
    w, r = tuned[t]["window"], tuned[t]
    n_prod = fpp.cv.production_n_estimators(r.get("best_iterations", []))

    fpp.artifacts.save_artifact(t, "window", {
        "L": int(w["L"]), "alpha": float(w["alpha"]),
        "L_venue": int(np.ceil(0.5 * w["L"])),
        "effective_window": w.get("effective_window"),
    }, VERSION)

    params = dict(r["params"])
    # The search scaffolding must not reach production. `n_estimators` is a search
    # ceiling and is replaced by the early-stopped production count; `max_bin` is
    # now tuned, so it is allowed through -- but only because something chose it.
    params["n_estimators"] = n_prod
    fpp.artifacts.save_artifact(t, "params", {
        "params": params,
        "n_estimators_production": n_prod,
        "dispersion": r.get("dispersion", {}),
    }, VERSION)

    fpp.artifacts.save_artifact(t, "cv", {"logloss": r["logloss"],
                                          "logloss_se": r.get("logloss_se")}, VERSION)
    fpp.artifacts.write_provenance(t, VERSION, n_rows=len(df), data_hash="clean", extra={
        "tuning_rounds": r.get("rounds", []),
        "converged": r.get("converged"),
        "tuning_folds": r.get("folds"),
        "holdout": holdout.get(t),
    })
    print(f"  {t:8} L={w['L']:>3} alpha={w['alpha']:.2f}  max_bin={params.get('max_bin')}  "
          f"lr={params.get('learning_rate')}  trees={n_prod}")

fpp.artifacts.freeze(VERSION)

  goals    L= 41 alpha=0.02  max_bin=64  lr=0.05  trees=417
  shots    L= 50 alpha=0.10  max_bin=64  lr=0.05  trees=170
  sot      L= 60 alpha=0.05  max_bin=64  lr=0.05  trees=101
  corners  L= 35 alpha=0.05  max_bin=64  lr=0.05  trees=102
Froze v2026-08-25 as current (4 target families)
  note: 1 version dir(s) on disk are not in the manifest: ['v2026-08-24']
        left in place -- delete them if they are stale.


'v2026-08-25'